# Spark Tune - Databricks ML Pipeline Demo

End-to-end ML pipeline reading data from a Databricks catalog and demonstrating:

5. **pre-processing** - Data preprocessing for model training and insight generation

In [0]:
# !pip install databricks-feature-engineering

In [0]:
# %restart_python

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

---
## 1. Load Data from Databricks Catalog

In [0]:
CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"
TABLE_NAME = "hdfc_demo_bank_customers"

FEATURE_SCHEMA_NAME = "feature_store"
FEATURE_TABLE_NAME = "hdfc_demo_customer_transaction_featuretools"
TSFRESH_FEATURE_TABLE_NAME = "hdfc_demo_cust_trans_tsfresh"
AUTO_FEATURE_TABLE_NAME = f"{TABLE_NAME}_auto"


# # Table Names
# credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.credit_card_transactions"
credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

# ftool_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_featuretools"
# tsfresh_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_tsfresh"
# auto_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_auto"
ftool_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{FEATURE_TABLE_NAME}"
tsfresh_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{TSFRESH_FEATURE_TABLE_NAME}"
auto_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{AUTO_FEATURE_TABLE_NAME}"





# Reading Original Data
df = spark.read.table(credit_card_transactions_table_name)

print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
df.printSchema()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

fe = FeatureEngineeringClient()

# Get feature columns for each table (excluding lookup keys and contact_timestamp)
ftool_df = spark.read.table(ftool_feature_table_name)
ftool_features = [col for col in ftool_df.columns if col not in ['customer_id', 'contact_timestamp']]

tsfresh_df = spark.read.table(tsfresh_feature_table_name)
tsfresh_features = [col for col in tsfresh_df.columns if col not in ['customer_id']]

auto_df = spark.read.table(auto_feature_table_name)
auto_features = [col for col in auto_df.columns if col not in ['customer_id', 'contact_timestamp']]

feature_lookups = [
    FeatureLookup(
        table_name=ftool_feature_table_name,
        lookup_key='customer_id',
        feature_names=ftool_features,
    ),
    FeatureLookup(
        table_name=tsfresh_feature_table_name,
        lookup_key='customer_id',
        feature_names=tsfresh_features,
    ),
    FeatureLookup(
        table_name=auto_feature_table_name,
        lookup_key='customer_id',
        timestamp_lookup_key='contact_timestamp',
        feature_names=auto_features,
    ),
                   
]

ML Flow Registry

In [0]:
import mlflow
mlflow.set_registry_uri("hdfc_cust_trans_demo")

In [0]:
# End any existing runs (in the case this notebook is being run for a second time)
mlflow.end_run()

# Start an mlflow run, which is needed to log the model
mlflow.start_run()

# Since the timestamp columns would likely cause the model to overfit the data
# unless additional feature engineering was performed, exclude them to avoid training on them.
# exclude_columns = ["trans_num", "trans_date_trans_time"]
exclude_columns = ["customer_id", "contact_timestamp"]

# Create the training set that includes the raw input data merged with corresponding features from both feature tables
training_set = fe.create_training_set(
    df=df,
    feature_lookups=feature_lookups,
    label="responded",
    exclude_columns=exclude_columns,
)

# Load the TrainingSet into a dataframe which can be passed into sklearn for training a model
training_df = training_set.load_df()

In [0]:
# Display the training dataframe, and note that it contains both the raw input data and the features from the feature tables, like `dropoff_is_weekend`
display(training_df.limit(5))

In [0]:
train_df, test_df = training_df.randomSplit([0.8, 0.2], seed=42)
display(train_df)

## Training Model

In [0]:
from databricks import automl

mlflow.end_run()
summary = automl.classify(train_df, target_col="responded", timeout_minutes=120)

In [0]:
help(summary)

In [0]:
supported_cols = ["annual_income_add_days_since_prev_contact", "has_mobile_app_div_emp_variation_rate", "email_click_rate_sub_consumer_price_index", "has_personal_loan_add_contacts_this_campaign", "num_products_sub_days_since_prev_contact", "account_balance_+_days_since_prev_contact", "age_div_contacts_this_campaign", "has_online_banking_add_euribor_3m_rate", "website_visits_last_30d_binned", "credit_score_add_euribor_3m_rate", "account_balance_div_consumer_price_index", "annual_income_sub_euribor_3m_rate", "email_open_rate_div_consumer_price_index", "age_div_email_open_rate", "email_click_rate_sub_website_visits_last_30d", "contacts_this_campaign_div_consumer_confidence_index", "age_sub_annual_income", "credit_score_sub_has_online_banking", "has_mortgage_sub_has_personal_loan", "has_personal_loan_sub_emp_variation_rate", "has_mortgage_add_nr_employed", "annual_income_sqrt", "min_transactions_amount_", "customer_tenure_years_div_euribor_3m_rate", "email_open_rate_mult_consumer_price_index", "has_online_banking_mult_email_open_rate", "age_+_has_mortgage", "email_click_rate_div_nr_employed", "emp_variation_rate_add_nr_employed", "account_balance_mult_contacts_prev_campaigns", "email_open_rate_sub_website_visits_last_30d", "annual_income_div_consumer_confidence_index", "has_personal_loan_div_consumer_confidence_index", "credit_score_add_has_credit_card", "age_div_email_click_rate", "has_credit_card_add_contacts_this_campaign", "age_add_consumer_confidence_index", "email_open_rate_cube", "has_mortgage_add_emp_variation_rate", "euribor_3m_rate_log", "has_credit_card_mult_email_click_rate", "email_open_rate_square", "annual_income_add_account_balance", "account_balance_mult_consumer_price_index", "age_sub_num_products", "contacts_prev_campaigns_sub_has_online_banking", "customer_tenure_years_sub_contacts_this_campaign", "customer_tenure_years_div_has_mortgage", "account_balance_add_website_visits_last_30d", "email_click_rate_sub_emp_variation_rate", "consumer_price_index_sub_euribor_3m_rate", "credit_score_binned", "credit_score_add_num_products", "annual_income_mult_euribor_3m_rate", "has_mortgage_mult_days_since_prev_contact", "has_personal_loan_add_email_open_rate", "num_products_add_emp_variation_rate", "website_visits_last_30d_add_nr_employed", "num_products_div_consumer_confidence_index", "has_credit_card_mult_num_products", "consumer_confidence_index_mult_nr_employed", "emp_variation_rate_sqrt", "account_balance_+_contacts_this_campaign", "account_balance_sub_emp_variation_rate", "age_sub_nr_employed", "has_credit_card_mult_nr_employed", "has_credit_card_div_consumer_confidence_index", "age_add_contacts_this_campaign", "contacts_prev_campaigns_binned", "age_cube", "consumer_price_index_binned", "annual_income_div_customer_tenure_years", "account_balance_add_email_click_rate", "email_open_rate_add_emp_variation_rate", "email_click_rate_add_emp_variation_rate", "age_div_has_mortgage", "contacts_prev_campaigns_div_consumer_price_index", "credit_score_sub_has_mobile_app", "has_personal_loan_mult_website_visits_last_30d", "contacts_this_campaign_mult_euribor_3m_rate", "days_since_prev_contact_sub_consumer_confidence_index", "customer_tenure_years_sub_days_since_prev_contact", "contacts_this_campaign_sub_has_online_banking", "contacts_prev_campaigns_sub_emp_variation_rate", "contacts_prev_campaigns_mult_has_online_banking", "email_click_rate_mult_euribor_3m_rate", "has_personal_loan_sub_nr_employed", "account_balance_mult_email_open_rate", "has_online_banking_div_nr_employed", "age_div_annual_income", "account_balance_mult_website_visits_last_30d", "num_products_add_has_mobile_app", "account_balance_mult_has_online_banking", "contacts_this_campaign_add_website_visits_last_30d", "has_online_banking_div_website_visits_last_30d", "has_mortgage_add_website_visits_last_30d", "age_mult_euribor_3m_rate", "has_online_banking_add_website_visits_last_30d", "annual_income_div_has_personal_loan", "email_click_rate_mult_consumer_price_index", "account_balance_mult_emp_variation_rate", "website_visits_last_30d_mult_consumer_confidence_index", "has_mobile_app_sub_consumer_price_index", "has_mobile_app_div_consumer_price_index", "customer_tenure_years_add_has_credit_card", "has_credit_card_mult_contacts_prev_campaigns", "num_products_div_consumer_price_index", "amount__value__sum_values", "credit_score_add_email_open_rate", "credit_score_div_has_online_banking", "account_balance_+_email_click_rate", "has_mortgage_mult_has_mobile_app", "days_since_prev_contact_add_consumer_confidence_index", "age_+_emp_variation_rate", "credit_score_sub_has_mortgage", "has_credit_card_add_contacts_prev_campaigns", "has_personal_loan_mult_has_credit_card", "has_personal_loan_div_contacts_this_campaign", "has_personal_loan_add_emp_variation_rate", "customer_tenure_years_sub_has_mobile_app", "has_online_banking_mult_website_visits_last_30d", "num_products_mult_website_visits_last_30d", "email_open_rate_add_website_visits_last_30d", "customer_tenure_years_div_has_online_banking", "has_credit_card", "credit_score_mult_customer_tenure_years", "contacts_this_campaign_mult_days_since_prev_contact", "has_mortgage_div_contacts_this_campaign", "has_online_banking_div_has_mobile_app", "amount__value__length", "credit_score_div_nr_employed", "has_online_banking_sub_email_open_rate", "annual_income_+_consumer_price_index", "consumer_price_index_add_euribor_3m_rate", "has_mortgage_sub_contacts_prev_campaigns", "has_mortgage_div_num_products", "age_sub_credit_score", "has_credit_card_sub_days_since_prev_contact", "account_balance_cube", "website_visits_last_30d", "annual_income_mult_contacts_this_campaign", "age_mult_credit_score", "num_products_div_email_click_rate", "customer_tenure_years_mult_has_credit_card", "has_mobile_app_div_consumer_confidence_index", "has_mortgage_cube", "annual_income_add_consumer_price_index", "days_since_prev_contact_sub_euribor_3m_rate", "credit_score_add_has_mobile_app", "sum_transactions_amount_", "has_credit_card_sub_num_products", "email_open_rate_add_email_click_rate", "annual_income_mult_credit_score", "annual_income_mult_emp_variation_rate", "account_balance_+_contacts_prev_campaigns", "account_balance_mult_customer_tenure_years", "num_products_add_email_open_rate", "contacts_prev_campaigns_sub_consumer_confidence_index", "has_credit_card_mult_has_online_banking", "has_mobile_app_log", "days_since_prev_contact_sub_has_online_banking", "email_open_rate_mult_email_click_rate", "num_products_add_days_since_prev_contact", "has_personal_loan_mult_contacts_prev_campaigns", "age_sub_has_credit_card", "annual_income_binned", "max_transactions_balance_after_txn_", "has_mobile_app_sub_emp_variation_rate", "credit_score_add_contacts_this_campaign", "days_since_prev_contact_add_nr_employed", "account_balance_+_email_open_rate", "max_transactions_amount_", "contacts_this_campaign_div_consumer_price_index", "has_credit_card_cube", "annual_income_mult_has_credit_card", "euribor_3m_rate_binned", "contacts_this_campaign_sub_email_open_rate", "days_since_prev_contact_div_email_click_rate", "contact_timestamp_dayofyear", "nr_employed_log", "has_credit_card_sub_contacts_prev_campaigns", "num_products_add_consumer_price_index", "consumer_confidence_index_square", "contacts_prev_campaigns_add_emp_variation_rate", "credit_score_sub_emp_variation_rate", "has_mobile_app_sub_email_click_rate", "credit_score_div_has_mortgage", "account_balance_mult_contacts_this_campaign", "contacts_prev_campaigns_div_has_mobile_app", "has_mobile_app_div_euribor_3m_rate", "contacts_this_campaign_add_email_click_rate", "age_sub_email_open_rate", "has_personal_loan_add_website_visits_last_30d", "email_open_rate_mult_website_visits_last_30d", "consumer_confidence_index_mult_euribor_3m_rate", "days_since_prev_contact_mult_consumer_confidence_index", "contact_timestamp_quarter", "has_personal_loan_sub_euribor_3m_rate", "age_add_contacts_prev_campaigns", "annual_income_mult_days_since_prev_contact", "days_since_prev_contact_mult_has_online_banking", "has_mortgage_mult_consumer_price_index", "days_since_prev_contact_div_website_visits_last_30d", "num_products", "account_balance_sub_credit_score", "customer_tenure_years_mult_consumer_price_index", "has_personal_loan_sub_has_mobile_app", "age_+_num_products", "has_personal_loan_div_has_mobile_app", "account_balance_div_days_since_prev_contact", "credit_score_mult_has_mortgage", "email_open_rate_div_website_visits_last_30d", "has_mortgage_div_days_since_prev_contact", "age_mult_email_open_rate", "customer_tenure_years_sub_email_click_rate", "account_balance_add_has_credit_card", "contacts_prev_campaigns_mult_consumer_confidence_index", "credit_score_add_email_click_rate", "contacts_this_campaign_square", "credit_score_sub_consumer_confidence_index", "age_+_contacts_prev_campaigns", "age_sub_euribor_3m_rate", "account_balance_+_nr_employed", "age_add_has_mobile_app", "email_click_rate_log", "num_products_add_consumer_confidence_index", "has_personal_loan_log", "contacts_this_campaign_mult_emp_variation_rate", "has_credit_card_div_has_mobile_app", "has_mortgage_sub_emp_variation_rate", "days_since_prev_contact_add_has_online_banking", "age_div_credit_score", "annual_income_sub_emp_variation_rate", "contact_timestamp_month", "email_click_rate", "account_balance_add_has_online_banking", "credit_score_add_customer_tenure_years", "customer_tenure_years_mult_emp_variation_rate", "contacts_prev_campaigns_add_nr_employed", "email_open_rate_add_consumer_price_index", "email_click_rate_div_euribor_3m_rate", "has_mortgage_sub_email_click_rate", "has_online_banking_log", "has_credit_card_div_days_since_prev_contact", "annual_income_sub_has_personal_loan", "has_credit_card_div_email_click_rate", "age_add_euribor_3m_rate", "age_mult_has_credit_card", "age_square", "contacts_prev_campaigns_mult_euribor_3m_rate", "customer_tenure_years_div_consumer_price_index", "contact_timestamp_is_weekend", "credit_score_mult_euribor_3m_rate", "consumer_confidence_index_sub_nr_employed", "email_open_rate_div_email_click_rate", "email_open_rate_div_consumer_confidence_index", "sum_transactions_balance_after_txn_", "website_visits_last_30d_sub_consumer_confidence_index", "has_mobile_app_mult_nr_employed", "has_online_banking_sub_email_click_rate", "has_mortgage_add_consumer_price_index", "contacts_this_campaign_sub_nr_employed", "account_balance_div_customer_tenure_years", "has_online_banking_mult_emp_variation_rate", "age_div_website_visits_last_30d", "account_balance_sub_website_visits_last_30d", "website_visits_last_30d_div_emp_variation_rate", "credit_score_div_website_visits_last_30d", "has_personal_loan_mult_euribor_3m_rate", "days_since_prev_contact_mult_website_visits_last_30d", "contacts_prev_campaigns_mult_emp_variation_rate", "account_balance_add_has_personal_loan", "customer_tenure_years_sub_has_mortgage", "account_balance_sub_euribor_3m_rate", "has_credit_card_mult_euribor_3m_rate", "num_products_div_contacts_prev_campaigns", "email_open_rate_mult_consumer_confidence_index", "sum_transactions_is_flagged_", "age_+_responded", "contacts_prev_campaigns_div_consumer_confidence_index", "account_balance_sub_consumer_price_index", "credit_score_mult_consumer_price_index", "has_personal_loan_square", "email_open_rate_binned", "website_visits_last_30d_add_emp_variation_rate", "has_mortgage_div_consumer_price_index", "age_mult_emp_variation_rate", "credit_score_div_has_personal_loan", "has_credit_card_sub_consumer_confidence_index", "has_personal_loan_mult_email_open_rate", "num_products_mult_days_since_prev_contact", "annual_income_div_has_mobile_app", "contacts_this_campaign_mult_email_click_rate", "credit_score_sub_days_since_prev_contact", "has_mortgage_sub_website_visits_last_30d", "has_personal_loan_div_nr_employed", "age_sub_days_since_prev_contact", "annual_income_square", "annual_income_add_email_open_rate", "email_click_rate_square", "customer_tenure_years_sub_has_personal_loan", "consumer_price_index_sub_nr_employed", "account_balance_sub_customer_tenure_years", "consumer_confidence_index_add_nr_employed", "mean_transactions_is_flagged_", "days_since_prev_contact_cube", "credit_score_mult_consumer_confidence_index", "days_since_prev_contact_div_emp_variation_rate", "has_personal_loan_mult_num_products", "mean_transactions_amount_", "city", "has_personal_loan_mult_consumer_confidence_index", "num_products_sub_contacts_this_campaign", "std_transactions_amount_", "age_add_num_products", "annual_income_+_credit_score", "num_products_add_website_visits_last_30d", "age_add_annual_income", "credit_score_div_contacts_this_campaign", "customer_tenure_years_add_contacts_prev_campaigns", "account_balance_add_emp_variation_rate", "credit_score_add_contacts_prev_campaigns", "contacts_prev_campaigns_div_website_visits_last_30d", "age_mult_website_visits_last_30d", "annual_income_mult_has_personal_loan", "account_balance_div_has_mobile_app", "amount__value__standard_deviation", "annual_income_div_email_open_rate", "annual_income_div_days_since_prev_contact", "customer_tenure_years_mult_has_mortgage", "account_balance_+_website_visits_last_30d", "account_balance_add_euribor_3m_rate", "annual_income_div_contacts_prev_campaigns", "consumer_price_index_log", "has_mortgage_mult_email_open_rate", "credit_score_mult_website_visits_last_30d", "credit_score_mult_has_online_banking", "email_open_rate_add_euribor_3m_rate", "has_credit_card_div_nr_employed", "emp_variation_rate_sub_nr_employed", "num_products_mult_has_online_banking", "contacts_prev_campaigns_mult_has_mobile_app", "has_credit_card_div_num_products", "has_mobile_app_add_email_open_rate", "customer_tenure_years_sub_email_open_rate", "customer_tenure_years_mult_contacts_prev_campaigns", "amount__value__mean", "has_online_banking_mult_has_mobile_app", "days_since_prev_contact_div_has_mobile_app", "emp_variation_rate_div_consumer_price_index", "std_transactions_is_flagged_", "email_click_rate_div_website_visits_last_30d", "annual_income_log", "has_mortgage_div_has_mobile_app", "annual_income_add_euribor_3m_rate", "contacts_this_campaign_sub_email_click_rate", "annual_income_sub_has_mobile_app", "has_mortgage_mult_has_personal_loan", "customer_tenure_years_log", "age_div_consumer_price_index", "has_credit_card_div_emp_variation_rate", "customer_tenure_years_div_website_visits_last_30d", "age_add_credit_score", "age_sub_has_mortgage", "age_add_has_online_banking", "contacts_prev_campaigns_div_email_open_rate", "has_mortgage_add_has_online_banking", "num_products_sub_euribor_3m_rate", "account_balance_add_customer_tenure_years", "days_since_prev_contact_mult_nr_employed", "has_credit_card_div_contacts_prev_campaigns", "days_since_prev_contact_log", "days_since_prev_contact_mult_contacts_prev_campaigns", "has_mortgage_add_email_click_rate", "nr_employed_cube", "annual_income_sub_has_credit_card", "num_unique_transactions_channel_", "age_add_email_click_rate", "contacts_this_campaign_sub_website_visits_last_30d", "consumer_confidence_index_sub_euribor_3m_rate", "account_balance_+_has_online_banking", "num_products_div_nr_employed", "credit_score_div_consumer_confidence_index", "account_balance_add_num_products", "days_since_prev_contact_mult_euribor_3m_rate", "annual_income_div_consumer_price_index", "num_products_add_contacts_this_campaign", "account_balance_add_email_open_rate", "contacts_this_campaign_binned", "contacts_prev_campaigns_mult_email_open_rate", "annual_income_mult_contacts_prev_campaigns", "has_personal_loan_sub_consumer_confidence_index", "days_since_prev_contact_add_consumer_price_index", "account_balance_sub_has_mobile_app", "account_balance_+_num_products", "annual_income_add_emp_variation_rate", "contacts_this_campaign_add_consumer_price_index", "consumer_price_index", "has_online_banking_mult_nr_employed", "account_balance_div_contacts_this_campaign", "account_balance_add_consumer_confidence_index", "annual_income_add_has_personal_loan", "emp_variation_rate_mult_consumer_confidence_index", "has_personal_loan_div_has_online_banking", "email_open_rate_add_nr_employed", "age_sub_email_click_rate", "email_open_rate", "has_mortgage_mult_num_products", "has_credit_card_div_contacts_this_campaign", "has_personal_loan_mult_consumer_price_index", "has_personal_loan_add_consumer_price_index", "has_credit_card_mult_emp_variation_rate", "num_products_sub_email_click_rate", "days_since_prev_contact_div_nr_employed", "has_online_banking_div_euribor_3m_rate", "contacts_this_campaign_add_consumer_confidence_index", "has_online_banking_add_has_mobile_app", "age_sub_has_personal_loan", "customer_tenure_years_div_contacts_this_campaign", "customer_tenure_years_mult_website_visits_last_30d", "age_mult_has_mortgage", "has_personal_loan_add_num_products", "customer_tenure_years_mult_nr_employed", "euribor_3m_rate_add_nr_employed", "has_personal_loan_div_contacts_prev_campaigns", "account_balance_sub_has_credit_card", "contacts_this_campaign_add_nr_employed", "has_online_banking_mult_consumer_price_index", "has_mortgage_mult_consumer_confidence_index", "account_balance_sub_email_click_rate", "has_personal_loan_sqrt", "has_personal_loan_sub_days_since_prev_contact", "age_+_has_credit_card", "has_mobile_app_sub_consumer_confidence_index", "credit_score_mult_emp_variation_rate", "age_+_credit_score", "credit_score_mult_has_mobile_app", "contacts_this_campaign_div_emp_variation_rate", "has_mortgage_mult_emp_variation_rate", "emp_variation_rate", "credit_score_sub_nr_employed", "account_balance_mult_euribor_3m_rate", "has_personal_loan_add_email_click_rate", "consumer_confidence_index", "website_visits_last_30d_square", "has_mobile_app_sqrt", "contacts_this_campaign_div_has_mobile_app", "has_online_banking_div_consumer_price_index", "has_credit_card_div_has_online_banking", "has_credit_card_mult_has_mobile_app", "emp_variation_rate_add_consumer_confidence_index", "amount__value__median", "customer_tenure_years_div_has_mobile_app", "email_click_rate_binned", "customer_tenure_years_add_consumer_price_index", "email_open_rate_sub_euribor_3m_rate", "age_+_consumer_price_index", "has_personal_loan_sub_contacts_this_campaign", "consumer_price_index_cube", "customer_tenure_years_add_email_click_rate", "days_since_prev_contact", "customer_tenure_years_sqrt", "customer_tenure_years_mult_has_online_banking", "has_credit_card_add_nr_employed", "contacts_prev_campaigns_div_euribor_3m_rate", "annual_income_add_has_credit_card", "has_credit_card_add_has_mobile_app", "contacts_this_campaign_mult_has_online_banking", "age_mult_consumer_price_index", "contacts_prev_campaigns_sub_nr_employed", "customer_tenure_years_sub_nr_employed", "has_mortgage_add_has_personal_loan", "annual_income_sub_num_products", "annual_income_sub_days_since_prev_contact", "customer_tenure_years_mult_email_click_rate", "consumer_confidence_index_binned", "customer_tenure_years_sub_num_products", "has_personal_loan_sub_consumer_price_index", "account_balance_mult_nr_employed", "has_mobile_app_mult_consumer_confidence_index", "customer_tenure_years_div_email_open_rate", "age_mult_num_products", "num_products_mult_email_click_rate", "has_mobile_app_mult_email_open_rate", "credit_score_add_has_online_banking", "contacts_prev_campaigns_add_has_online_banking", "has_credit_card_sub_email_open_rate", "email_click_rate_add_euribor_3m_rate", "website_visits_last_30d_add_consumer_confidence_index", "has_mortgage_log", "credit_score_sub_contacts_prev_campaigns", "customer_tenure_years_square", "num_products_mult_consumer_price_index", "email_click_rate_sub_nr_employed", "has_personal_loan_add_contacts_prev_campaigns", "euribor_3m_rate_sub_nr_employed", "has_credit_card_mult_consumer_price_index", "account_balance_add_consumer_price_index", "account_balance_+_responded", "emp_variation_rate_log", "contacts_prev_campaigns_add_consumer_price_index", "age_+_annual_income", "credit_score_add_website_visits_last_30d", "account_balance_sub_has_online_banking", "annual_income_div_email_click_rate", "contacts_this_campaign_sub_has_mobile_app", "has_mobile_app_div_nr_employed", "account_balance_+_annual_income", "email_click_rate_sqrt", "account_balance_div_has_online_banking", "days_since_prev_contact_sub_nr_employed", "num_products_sub_has_online_banking", "num_products_mult_contacts_prev_campaigns", "customer_tenure_years_sub_has_online_banking", "age_+_has_mobile_app", "age_sub_website_visits_last_30d", "annual_income_mult_website_visits_last_30d", "has_mortgage_sub_consumer_price_index", "account_balance_div_credit_score", "customer_tenure_years_add_website_visits_last_30d", "has_mobile_app_div_email_click_rate", "age_div_has_mobile_app", "has_online_banking_mult_euribor_3m_rate", "num_products_cube", "customer_tenure_years_sub_has_credit_card", "age_+_email_click_rate", "contacts_prev_campaigns_sub_consumer_price_index", "contacts_this_campaign_mult_email_open_rate", "age_sub_emp_variation_rate", "contacts_this_campaign_add_days_since_prev_contact", "has_online_banking_add_consumer_confidence_index", "num_products_sub_has_mobile_app", "email_click_rate_div_emp_variation_rate", "age_+_website_visits_last_30d", "has_mortgage_add_num_products", "customer_tenure_years_sub_euribor_3m_rate", "num_products_sub_consumer_price_index", "has_mortgage", "contacts_this_campaign_mult_nr_employed", "has_personal_loan_cube", "customer_tenure_years_div_email_click_rate", "contacts_prev_campaigns_add_has_mobile_app", "website_visits_last_30d_div_consumer_confidence_index", "age_add_consumer_price_index", "contacts_prev_campaigns_sub_website_visits_last_30d", "has_personal_loan", "emp_variation_rate_mult_consumer_price_index", "days_since_prev_contact_mult_email_open_rate", "account_balance_+_consumer_confidence_index", "annual_income_div_nr_employed", "age_add_website_visits_last_30d", "contacts_prev_campaigns", "contacts_this_campaign", "customer_tenure_years_mult_num_products", "credit_score_div_customer_tenure_years", "age_sub_consumer_confidence_index", "customer_tenure_years_sub_website_visits_last_30d", "age_sub_contacts_this_campaign", "num_products_div_contacts_this_campaign", "credit_score_div_euribor_3m_rate", "has_mobile_app_add_euribor_3m_rate", "age_div_customer_tenure_years", "has_mortgage_add_days_since_prev_contact", "num_products_sub_emp_variation_rate", "num_products_add_email_click_rate", "days_since_prev_contact_div_euribor_3m_rate", "nr_employed_square", "contacts_this_campaign_log", "has_credit_card_div_email_open_rate", "has_online_banking_cube", "annual_income_mult_consumer_price_index", "has_online_banking_sqrt", "account_balance", "age_div_num_products", "annual_income_sub_website_visits_last_30d", "contacts_this_campaign_add_has_online_banking", "email_open_rate_sub_consumer_price_index", "num_products_sub_nr_employed", "annual_income_cube", "email_open_rate_div_emp_variation_rate", "emp_variation_rate_mult_nr_employed", "num_products_mult_contacts_this_campaign", "age_sub_has_online_banking", "age_add_customer_tenure_years", "contact_timestamp_weekofyear", "credit_score_add_days_since_prev_contact", "contacts_this_campaign_mult_website_visits_last_30d", "age", "has_mortgage_div_contacts_prev_campaigns", "age_mult_email_click_rate", "has_mobile_app_add_nr_employed", "has_mobile_app_square", "contacts_this_campaign_sub_emp_variation_rate", "consumer_price_index_mult_consumer_confidence_index", "account_balance_+_has_mobile_app", "account_balance_mult_credit_score", "age_add_has_credit_card", "age_div_account_balance", "annual_income_mult_has_online_banking", "customer_tenure_years_add_days_since_prev_contact", "account_balance_log", "account_balance_sub_num_products", "annual_income_sub_has_online_banking", "credit_score_sub_customer_tenure_years", "account_balance_add_has_mortgage", "num_products_sqrt", "annual_income_+_consumer_confidence_index", "has_credit_card_div_website_visits_last_30d", "website_visits_last_30d_sub_euribor_3m_rate", "annual_income_sub_contacts_prev_campaigns", "email_open_rate_sub_email_click_rate", "annual_income_div_euribor_3m_rate", "has_credit_card_add_has_online_banking", "num_products_sub_contacts_prev_campaigns", "euribor_3m_rate_square", "has_mobile_app_mult_euribor_3m_rate", "has_mortgage_sub_nr_employed", "has_mobile_app_sub_euribor_3m_rate", "has_mobile_app_cube", "num_products_mult_consumer_confidence_index", "num_products_sub_email_open_rate", "annual_income_div_has_online_banking", "has_credit_card_mult_contacts_this_campaign", "has_online_banking_sub_consumer_confidence_index", "customer_tenure_years_div_consumer_confidence_index", "credit_score_sqrt", "has_mortgage_add_email_open_rate", "credit_score_sub_has_personal_loan", "has_mortgage_mult_website_visits_last_30d", "has_mortgage_sub_contacts_this_campaign", "contacts_this_campaign_mult_has_mobile_app", "consumer_price_index_mult_euribor_3m_rate", "customer_tenure_years_div_has_credit_card", "has_mortgage_sub_has_online_banking", "num_products_add_nr_employed", "has_credit_card_add_email_open_rate", "credit_score_sub_has_credit_card", "account_balance_add_contacts_this_campaign", "credit_score_div_days_since_prev_contact", "has_credit_card_add_euribor_3m_rate", "contacts_prev_campaigns_add_website_visits_last_30d", "account_balance_div_consumer_confidence_index", "std_transactions_balance_after_txn_", "consumer_price_index_add_nr_employed", "account_balance_+_has_personal_loan", "consumer_confidence_index_div_euribor_3m_rate", "contacts_this_campaign_mult_contacts_prev_campaigns", "credit_score_mult_days_since_prev_contact", "days_since_prev_contact_div_consumer_confidence_index", "contacts_prev_campaigns_sub_email_open_rate", "account_balance_sub_has_personal_loan", "has_online_banking_mult_email_click_rate", "has_personal_loan_div_email_click_rate", "has_credit_card_sub_emp_variation_rate", "has_personal_loan_div_website_visits_last_30d", "annual_income_mult_consumer_confidence_index", "customer_tenure_years_div_contacts_prev_campaigns", "consumer_price_index_sub_consumer_confidence_index", "days_since_prev_contact_add_has_mobile_app", "has_mobile_app_add_email_click_rate", "account_balance_add_days_since_prev_contact", "website_visits_last_30d_sub_consumer_price_index", "annual_income_add_credit_score", "emp_variation_rate_binned", "has_personal_loan_div_days_since_prev_contact", "age_div_contacts_prev_campaigns", "has_mortgage_sqrt", "annual_income_div_emp_variation_rate", "days_since_prev_contact_sub_website_visits_last_30d", "has_mortgage_add_contacts_this_campaign", "contacts_prev_campaigns_mult_email_click_rate", "euribor_3m_rate_cube", "has_personal_loan_sub_website_visits_last_30d", "account_balance_sub_contacts_this_campaign", "credit_score_div_emp_variation_rate", "has_mortgage_add_consumer_confidence_index", "credit_score_sub_num_products", "amount__value__maximum", "customer_tenure_years_mult_contacts_this_campaign", "annual_income_div_website_visits_last_30d", "contacts_this_campaign_add_contacts_prev_campaigns", "email_click_rate_div_consumer_confidence_index", "has_mortgage_mult_has_credit_card", "customer_tenure_years_mult_days_since_prev_contact", "days_since_prev_contact_add_contacts_prev_campaigns", "has_personal_loan_add_has_credit_card", "age_mult_consumer_confidence_index", "has_mortgage_sub_consumer_confidence_index", "emp_variation_rate_sub_consumer_confidence_index", "annual_income_sub_nr_employed", "has_credit_card_add_consumer_price_index", "customer_tenure_years_cube", "contacts_this_campaign_sub_days_since_prev_contact", "age_div_has_online_banking", "contacts_this_campaign_sqrt", "customer_tenure_years_add_has_mobile_app", "account_balance_+_credit_score", "has_personal_loan_add_consumer_confidence_index", "has_personal_loan_div_euribor_3m_rate", "has_personal_loan_div_consumer_price_index", "emp_variation_rate_mult_euribor_3m_rate", "has_mobile_app", "account_balance_+_emp_variation_rate", "annual_income_add_consumer_confidence_index", "email_click_rate_sub_euribor_3m_rate", "customer_tenure_years_add_contacts_this_campaign", "has_mobile_app_add_emp_variation_rate", "has_mortgage_square", "annual_income_div_num_products", "email_click_rate_div_consumer_price_index", "contacts_this_campaign_mult_consumer_price_index", "annual_income_mult_account_balance", "amount__value__variance", "has_personal_loan_sub_email_open_rate", "has_personal_loan_sub_num_products", "days_since_prev_contact_add_email_open_rate", "contacts_prev_campaigns_add_email_open_rate", "account_balance_sub_has_mortgage", "has_mobile_app_add_website_visits_last_30d", "has_personal_loan_sub_contacts_prev_campaigns", "has_mortgage_sub_has_mobile_app", "annual_income_sub_has_mortgage", "num_unique_transactions_transaction_type_", "age_mult_nr_employed", "days_since_prev_contact_add_euribor_3m_rate", "account_balance_add_nr_employed", "num_products_div_days_since_prev_contact", "credit_score_div_contacts_prev_campaigns", "customer_tenure_years_add_consumer_confidence_index", "has_online_banking_square", "has_credit_card_div_consumer_price_index", "age_div_has_personal_loan", "has_online_banking_sub_website_visits_last_30d", "has_credit_card_sub_consumer_price_index", "age_+_euribor_3m_rate", "credit_score_cube", "website_visits_last_30d_sub_nr_employed", "has_personal_loan_add_euribor_3m_rate", "age_add_has_mortgage", "account_balance_+_euribor_3m_rate", "contact_timestamp_day", "customer_tenure_years", "education", "age_add_account_balance", "has_personal_loan_div_num_products", "days_since_prev_contact_binned", "account_balance_div_has_personal_loan", "has_personal_loan_mult_nr_employed", "age_mult_customer_tenure_years", "account_balance_div_contacts_prev_campaigns", "customer_tenure_years_add_num_products", "amount__value__minimum", "account_balance_sub_consumer_confidence_index", "website_visits_last_30d_mult_consumer_price_index", "annual_income_div_contacts_this_campaign", "age_div_nr_employed", "credit_score_div_email_open_rate", "customer_tenure_years_mult_euribor_3m_rate", "annual_income_add_has_mortgage", "days_since_prev_contact_sub_has_mobile_app", "age_div_consumer_confidence_index", "annual_income_add_has_mobile_app", "contacts_prev_campaigns_add_consumer_confidence_index", "has_online_banking_sub_consumer_price_index", "days_since_prev_contact_sub_email_open_rate", "emp_variation_rate_div_euribor_3m_rate", "has_personal_loan_add_has_mobile_app", "has_personal_loan_mult_has_mobile_app", "days_since_prev_contact_sub_emp_variation_rate", "age_+_email_open_rate", "account_balance_div_has_credit_card", "account_balance_add_credit_score", "nr_employed_binned", "credit_score_mult_has_credit_card", "has_credit_card_sub_euribor_3m_rate", "contact_timestamp_year", "age_add_has_personal_loan", "days_since_prev_contact_add_email_click_rate", "has_mobile_app_mult_consumer_price_index", "email_open_rate_sub_consumer_confidence_index", "age_sub_customer_tenure_years", "account_balance_+_age", "has_online_banking_div_consumer_confidence_index", "age_add_nr_employed", "has_personal_loan_add_has_online_banking", "account_balance_mult_has_credit_card", "customer_tenure_years_add_has_online_banking", "days_since_prev_contact_add_emp_variation_rate", "euribor_3m_rate", "num_products_div_has_mobile_app", "contacts_this_campaign_add_euribor_3m_rate", "age_log", "email_click_rate_add_consumer_confidence_index", "credit_score_log", "credit_score_mult_email_open_rate", "account_balance_sub_email_open_rate", "has_mobile_app_sub_nr_employed", "annual_income_sub_email_open_rate", "annual_income_mult_has_mortgage", "num_products_log", "num_products_add_euribor_3m_rate", "annual_income_mult_email_click_rate", "has_mortgage_mult_nr_employed", "has_mobile_app_add_consumer_price_index", "age_+_contacts_this_campaign", "contacts_prev_campaigns_sub_email_click_rate", "account_balance_+_has_credit_card", "account_balance_sub_nr_employed", "email_open_rate_sub_nr_employed", "credit_score_mult_has_personal_loan", "has_mortgage_sub_euribor_3m_rate", "num_products_div_euribor_3m_rate", "num_products_div_email_open_rate", "customer_tenure_years_add_has_mortgage", "emp_variation_rate_div_consumer_confidence_index", "credit_score_sub_email_click_rate", "annual_income_add_has_online_banking", "nr_employed_sqrt", "credit_score_div_num_products", "has_credit_card_add_email_click_rate", "annual_income_add_nr_employed", "days_since_prev_contact_add_website_visits_last_30d", "age_mult_annual_income", "account_balance_mult_num_products", "emp_variation_rate_add_euribor_3m_rate", "contacts_this_campaign_div_contacts_prev_campaigns", "contacts_prev_campaigns_div_nr_employed", "has_credit_card_sqrt", "contacts_this_campaign_div_euribor_3m_rate", "contact_timestamp_dayofweek", "annual_income_sub_account_balance", "credit_score_div_email_click_rate", "days_since_prev_contact_sub_consumer_price_index", "age_add_emp_variation_rate", "credit_score_add_consumer_price_index", "consumer_price_index_add_consumer_confidence_index", "has_mortgage_div_has_online_banking", "euribor_3m_rate_div_nr_employed", "account_balance_div_email_open_rate", "has_mortgage_sub_email_open_rate", "age_sqrt", "website_visits_last_30d_cube", "account_balance_div_has_mortgage", "has_online_banking_sub_emp_variation_rate", "annual_income_sub_consumer_confidence_index", "has_mortgage_div_nr_employed", "has_mortgage_sub_num_products", "has_personal_loan_mult_emp_variation_rate", "consumer_confidence_index_cube", "account_balance_mult_consumer_confidence_index", "credit_score_mult_num_products", "prev_campaign_outcome", "consumer_price_index_div_euribor_3m_rate", "has_credit_card_sub_has_mobile_app", "contacts_this_campaign_div_has_online_banking", "account_balance_+_consumer_price_index", "has_mobile_app_sub_email_open_rate", "account_balance_+_has_mortgage", "annual_income_sub_customer_tenure_years", "annual_income_mult_has_mobile_app", "annual_income_add_customer_tenure_years", "age_sub_account_balance", "age_mult_contacts_this_campaign", "email_open_rate_div_nr_employed", "contacts_this_campaign_sub_consumer_confidence_index", "euribor_3m_rate_sqrt", "has_online_banking_sub_nr_employed", "age_+_nr_employed", "num_products_sub_consumer_confidence_index", "credit_score_sub_website_visits_last_30d", "has_online_banking_sub_euribor_3m_rate", "account_balance_div_num_products", "has_online_banking", "has_mortgage_div_email_click_rate", "days_since_prev_contact_div_has_online_banking", "age_+_consumer_confidence_index", "contact_method", "has_personal_loan_div_email_open_rate", "credit_score_sub_consumer_price_index", "account_balance_binned", "email_click_rate_add_consumer_price_index", "annual_income_+_customer_tenure_years", "annual_income_sub_contacts_this_campaign", "account_balance_sqrt", "account_balance_add_contacts_prev_campaigns", "account_balance_div_website_visits_last_30d", "contacts_prev_campaigns_div_emp_variation_rate", "days_since_prev_contact_sqrt", "annual_income_mult_customer_tenure_years", "has_credit_card_sub_contacts_this_campaign", "consumer_confidence_index_add_euribor_3m_rate", "amount__value__absolute_maximum", "age_+_has_online_banking", "annual_income_sub_consumer_price_index", "has_personal_loan_sub_has_credit_card", "email_open_rate_sub_emp_variation_rate", "customer_tenure_years_sub_consumer_price_index", "has_mortgage_mult_email_click_rate", "days_since_prev_contact_mult_consumer_price_index", "website_visits_last_30d_div_euribor_3m_rate", "contacts_this_campaign_sub_consumer_price_index", "age_add_days_since_prev_contact", "num_products_mult_email_open_rate", "has_mobile_app_mult_email_click_rate", "contacts_this_campaign_div_days_since_prev_contact", "has_personal_loan_div_emp_variation_rate", "has_personal_loan_div_has_credit_card", "days_since_prev_contact_mult_emp_variation_rate", "age_sub_consumer_price_index", "credit_score_mult_nr_employed", "customer_tenure_years_mult_email_open_rate", "annual_income_add_contacts_this_campaign", "customer_tenure_years_div_nr_employed", "num_unique_transactions_merchant_category_", "customer_tenure_years_div_emp_variation_rate", "has_credit_card_sub_has_online_banking", "customer_tenure_years_sub_emp_variation_rate", "contacts_prev_campaigns_div_has_online_banking", "credit_score_add_has_mortgage", "credit_score", "day_of_week", "credit_score_add_nr_employed", "has_personal_loan_add_days_since_prev_contact", "account_balance_square", "num_products_sub_website_visits_last_30d", "days_since_prev_contact_sub_email_click_rate", "has_mobile_app_sub_website_visits_last_30d", "has_online_banking_div_email_click_rate", "credit_score_add_has_personal_loan", "has_online_banking_mult_consumer_confidence_index", "account_balance_div_euribor_3m_rate", "email_open_rate_sqrt", "annual_income", "has_personal_loan_mult_has_online_banking", "count_transactions_", "credit_score_div_has_credit_card", "emp_variation_rate_div_nr_employed", "contacts_this_campaign_add_email_open_rate", "age_div_euribor_3m_rate", "email_click_rate_add_website_visits_last_30d", "days_since_prev_contact_mult_has_mobile_app", "age_div_has_credit_card", "customer_tenure_years_sub_contacts_prev_campaigns", "has_mortgage_div_euribor_3m_rate", "email_click_rate_mult_website_visits_last_30d", "num_products_div_website_visits_last_30d", "has_mobile_app_add_consumer_confidence_index", "customer_tenure_years_div_num_products", "annual_income_add_website_visits_last_30d", "annual_income_mult_nr_employed", "email_click_rate_mult_nr_employed", "credit_score_sub_euribor_3m_rate", "age_sub_has_mobile_app", "has_credit_card_mult_email_open_rate", "has_mortgage_mult_euribor_3m_rate", "occupation", "email_click_rate_mult_emp_variation_rate", "contacts_prev_campaigns_log", "website_visits_last_30d_add_consumer_price_index", "annual_income_add_num_products", "days_since_prev_contact_div_consumer_price_index", "emp_variation_rate_sub_consumer_price_index", "contacts_prev_campaigns_mult_website_visits_last_30d", "contacts_this_campaign_sub_euribor_3m_rate", "has_credit_card_sub_email_click_rate", "website_visits_last_30d_sub_emp_variation_rate", "has_credit_card_add_consumer_confidence_index", "has_mortgage_add_has_credit_card", "consumer_price_index_sqrt", "has_credit_card_mult_consumer_confidence_index", "age_mult_has_mobile_app", "age_add_email_open_rate", "customer_tenure_years_mult_has_mobile_app", "annual_income_mult_num_products", "credit_score_sub_email_open_rate", "account_balance_mult_has_mobile_app", "contacts_this_campaign_div_email_click_rate", "days_since_prev_contact_sub_contacts_prev_campaigns", "has_mortgage_div_emp_variation_rate", "has_credit_card_div_euribor_3m_rate", "email_open_rate_div_euribor_3m_rate", "has_mortgage_sub_has_credit_card", "has_online_banking_add_nr_employed", "age_binned", "has_credit_card_mult_days_since_prev_contact", "campaign_month", "has_mortgage_mult_contacts_prev_campaigns", "contacts_this_campaign_add_has_mobile_app", "gender", "annual_income_sub_credit_score", "has_mortgage_div_website_visits_last_30d", "num_products_square", "age_+_days_since_prev_contact", "customer_tenure_years_add_emp_variation_rate", "credit_score_mult_contacts_prev_campaigns", "nr_employed", "days_since_prev_contact_square", "account_balance_sub_contacts_prev_campaigns", "annual_income_mult_email_open_rate", "customer_tenure_years_div_days_since_prev_contact", "website_visits_last_30d_add_euribor_3m_rate", "age_mult_has_personal_loan", "email_open_rate_log", "account_balance_div_nr_employed", "email_open_rate_mult_euribor_3m_rate", "email_click_rate_sub_consumer_confidence_index", "customer_tenure_years_div_has_personal_loan", "has_online_banking_div_emp_variation_rate", "contacts_prev_campaigns_sqrt", "emp_variation_rate_add_consumer_price_index", "annual_income_div_has_credit_card", "num_products_mult_has_mobile_app", "has_online_banking_add_consumer_price_index", "days_since_prev_contact_div_email_open_rate", "contacts_this_campaign_div_nr_employed", "contacts_prev_campaigns_add_email_click_rate", "has_credit_card_sub_website_visits_last_30d", "account_balance_mult_email_click_rate", "account_balance_+_customer_tenure_years", "customer_tenure_years_binned", "website_visits_last_30d_div_consumer_price_index", "has_mortgage_mult_has_online_banking", "email_click_rate_mult_consumer_confidence_index", "has_credit_card_add_website_visits_last_30d", "customer_tenure_years_add_euribor_3m_rate", "emp_variation_rate_square", "website_visits_last_30d_log", "account_balance_mult_days_since_prev_contact", "min_transactions_balance_after_txn_", "contacts_prev_campaigns_add_euribor_3m_rate", "customer_tenure_years_add_has_personal_loan", "credit_score_square", "has_mobile_app_mult_website_visits_last_30d", "has_online_banking_add_email_open_rate", "has_mortgage_div_has_personal_loan", "has_mortgage_add_euribor_3m_rate", "account_balance_mult_has_mortgage", "consumer_price_index_square", "num_products_add_contacts_prev_campaigns", "has_mobile_app_div_email_open_rate", "annual_income_div_credit_score", "annual_income_add_contacts_prev_campaigns", "email_click_rate_add_nr_employed", "age_mult_account_balance", "customer_tenure_years_mult_consumer_confidence_index", "has_mortgage_div_email_open_rate", "annual_income_add_email_click_rate", "has_credit_card_sub_nr_employed", "mean_transactions_balance_after_txn_", "email_click_rate_cube", "customer_tenure_years_sub_consumer_confidence_index", "contacts_this_campaign_add_emp_variation_rate", "has_mortgage_sub_days_since_prev_contact", "credit_score_add_consumer_confidence_index", "contacts_prev_campaigns_square", "age_mult_contacts_prev_campaigns", "website_visits_last_30d_mult_euribor_3m_rate", "has_credit_card_add_num_products", "customer_tenure_years_add_nr_employed", "days_since_prev_contact_mult_email_click_rate", "contacts_prev_campaigns_cube", "account_balance_mult_has_personal_loan", "has_credit_card_square", "website_visits_last_30d_sqrt", "has_personal_loan_sub_email_click_rate", "contacts_this_campaign_div_website_visits_last_30d", "contacts_prev_campaigns_mult_nr_employed", "has_online_banking_sub_has_mobile_app", "num_products_binned", "euribor_3m_rate_mult_nr_employed", "age_mult_has_online_banking", "age_+_has_personal_loan", "num_products_div_has_online_banking", "age_sub_contacts_prev_campaigns", "age_div_days_since_prev_contact", "annual_income_div_account_balance", "has_online_banking_add_email_click_rate", "consumer_confidence_index_div_nr_employed", "num_products_mult_emp_variation_rate", "has_mobile_app_mult_emp_variation_rate", "days_since_prev_contact_div_contacts_prev_campaigns", "credit_score_sub_contacts_this_campaign", "has_personal_loan_sub_has_online_banking", "has_credit_card_add_days_since_prev_contact", "email_open_rate_add_consumer_confidence_index", "has_credit_card_log", "has_mobile_app_div_website_visits_last_30d", "annual_income_div_has_mortgage", "contacts_this_campaign_div_email_open_rate", "contacts_this_campaign_cube", "customer_tenure_years_mult_has_personal_loan", "age_mult_days_since_prev_contact", "has_credit_card_mult_website_visits_last_30d", "customer_tenure_years_add_email_open_rate", "has_credit_card_add_emp_variation_rate", "website_visits_last_30d_mult_emp_variation_rate", "emp_variation_rate_sub_euribor_3m_rate", "contacts_this_campaign_mult_consumer_confidence_index", "contacts_prev_campaigns_sub_has_mobile_app", "num_products_mult_nr_employed", "credit_score_mult_email_click_rate", "has_personal_loan_mult_days_since_prev_contact", "annual_income_sub_email_click_rate", "annual_income_+_contacts_this_campaign", "consumer_price_index_mult_nr_employed", "contacts_prev_campaigns_sub_euribor_3m_rate", "emp_variation_rate_cube", "has_personal_loan_mult_contacts_this_campaign", "account_balance_div_emp_variation_rate", "credit_score_add_emp_variation_rate", "age_+_customer_tenure_years", "has_mortgage_mult_contacts_this_campaign", "contacts_prev_campaigns_div_email_click_rate", "num_products_add_has_online_banking", "credit_score_mult_contacts_this_campaign", "has_mortgage_div_has_credit_card", "has_mortgage_div_consumer_confidence_index", "website_visits_last_30d_mult_nr_employed", "has_mortgage_add_has_mobile_app", "amount__value__root_mean_square", "has_mortgage_add_contacts_prev_campaigns", "email_open_rate_mult_nr_employed", "num_products_mult_euribor_3m_rate", "has_online_banking_add_emp_variation_rate", "has_personal_loan_add_nr_employed", "annual_income_+_contacts_prev_campaigns", "has_personal_loan_mult_email_click_rate", "has_online_banking_div_email_open_rate", "max_transactions_is_flagged_", "age_div_emp_variation_rate", "credit_score_div_has_mobile_app", "contacts_prev_campaigns_mult_consumer_price_index", "credit_score_div_consumer_price_index", "account_balance_sub_days_since_prev_contact", "account_balance_div_email_click_rate", "num_products_div_emp_variation_rate", "email_open_rate_mult_emp_variation_rate", "website_visits_last_30d_div_nr_employed", "marital_status", "consumer_price_index_div_nr_employed", "account_balance_add_has_mobile_app", "contacts_this_campaign_sub_contacts_prev_campaigns", "consumer_price_index_div_consumer_confidence_index"]

len(supported_cols)